# Multi-Agent Orchestration over MCP: One Governed Front Door for Any AI Client

**The question:** every AI client (Claude, ChatGPT, Cursor, an in-house assistant) wants to answer
questions from governed company data. How do you give all of them one front door, and keep the
answer for each person exactly what that person is entitled to, without trusting the client or the
model to enforce it?

**Already built by `./demo.sh setup` (`lab/setup.sql`):** Northwind's sales and support data, a
semantic view per domain, a fail-closed row access policy on sales, two caller's-rights bridge
procedures, a sales specialist, a support specialist and an orchestrator agent, and the persona
grants. Three persona users (global sales exec, EMEA seller, support rep) sign in with key pairs.

**In this notebook:**

| Section | What it does | CHECK |
|---|---|---|
| 0 | Connect, account guard | - |
| 1 | Inspect the governed stack: agents, grants, row access policy | agents exist, grants split by domain, policy returns 44 / 16 / 0 rows |
| 2 | Expose `ORCHESTRATOR_AGENT` as the Snowflake-managed MCP server `ORCHESTRATOR_MCP` | one `CORTEX_AGENT_RUN` tool, USAGE for each persona |
| 3 | The agent-toolset alternative: `TOOLSET_AGENT` behind `TOOLSET_MCP` | toolset agent and server exist, same tool name |
| 4 | The persona boundary over MCP: `tools/list` and `tools/call` as each persona on both servers, then the same question over SQL `DATA_AGENT_RUN` | same tool for all; global sees all theaters, EMEA sees EMEA only, support gets no sales numbers (over MCP and over SQL) |
| 5 | Evaluate both architectures on the same five questions | both runs complete with scores |
| 6 | The demo click-path | - |

Run it top to bottom with `./demo.sh build --yes`, step through it with `./demo.sh notebook`, or
open it in Snowsight Workspaces (Section 4 needs the persona keys, so it runs locally only).

In [ ]:
# Session: Snowsight Workspaces supplies one; a local kernel (./demo.sh notebook,
# ./demo.sh build) opens the connection saved in .env by ./demo.sh setup.
import json
import os
import re
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

DB = "MULTI_AGENT_MCP_DEMO"
SCHEMA = "APP"
FQ = f"{DB}.{SCHEMA}"
WH = "MULTI_AGENT_MCP_DEMO_WH"
ADMIN_ROLE = "MULTI_AGENT_MCP_DEMO_ADMIN"
PERSONA_ROLES = ["MULTI_AGENT_MCP_DEMO_SALES_GLOBAL", "MULTI_AGENT_MCP_DEMO_SALES_EMEA", "MULTI_AGENT_MCP_DEMO_SUPPORT"]
QUESTION = "What were total new bookings in FY25-Q4 by theater?"
# The account this lab may run in. ./demo.sh setup writes it to .env;
# in Workspaces, replace the placeholder with your demo account name.
DEMO_ACCOUNT = os.environ.get("DEMO_EXPECTED_ACCOUNT", "<YOUR_DEMO_ACCOUNT>")
SKIP_EVALS = os.environ.get("DEMO_SKIP_EVALS") == "1"

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    LOCAL = False
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.config(
        "connection_name", os.environ.get("SNOWFLAKE_DEFAULT_CONNECTION_NAME", "default")).create()
    LOCAL = True

if LOCAL:
    # Local kernels have no SQL cells: run each %%sql cell through this session instead.
    from IPython.core.magic import register_cell_magic
    from IPython.display import display

    @register_cell_magic
    def sql(line, cell):
        args = line.split()
        name = args[args.index("-r") + 1] if "-r" in args else None
        rows = session.sql(cell).collect()
        result = pd.DataFrame([r.as_dict() for r in rows])
        if name:
            get_ipython().user_ns[name] = result
        display(result.head(50))

CHECKS = {}


def check(section, ok, evidence):
    """Print one CHECK line (./demo.sh build echoes these) and stop on FAIL."""
    status = "PASS" if ok else "FAIL"
    CHECKS[section] = (status, evidence)
    print(f"CHECK {section}: {status} | {evidence}")
    if not ok:
        raise AssertionError(f"CHECK {section} failed: fix the cause and re-run this section")


def q(text):
    return session.sql(text).to_pandas()


def col(df, name):
    """SHOW output column, whatever case the driver returned it in."""
    return df[[c for c in df.columns if c.strip('"').lower() == name][0]]


account, user = session.sql("SELECT CURRENT_ACCOUNT_NAME(), CURRENT_USER()").collect()[0]
if account != DEMO_ACCOUNT:
    raise SystemExit(f"STOP: connected to {account}, but this lab may only run in {DEMO_ACCOUNT}. "
                     "Set DEMO_EXPECTED_ACCOUNT (./demo.sh setup does) or edit DEMO_ACCOUNT above.")
if not session.sql(f"SHOW DATABASES LIKE '{DB}'").collect():
    raise SystemExit(f"STOP: {DB} does not exist yet. Run ./demo.sh setup (or lab/setup.sql) first.")

session.use_role(ADMIN_ROLE)
session.use_warehouse(WH)
session.sql(f"USE SCHEMA {FQ}").collect()
print(f"Connected as {user} / role {ADMIN_ROLE} / account {account} / "
      f"{'local kernel' if LOCAL else 'Workspaces session'}")

---
## Section 1 - The governed stack behind the front door

Four layers decide what each persona gets. None of them is a prompt:

1. **USAGE on the orchestrator and both bridges**: everyone may ask.
2. **USAGE on a specialist agent**: the bridge's nested `DATA_AGENT_RUN` runs as the caller, so a
   persona without USAGE on `SALES_AGENT` is refused by Snowflake (error 399513, relayed as `NOT_ENTITLED`).
3. **SELECT on the semantic view and its base table**: the specialist's Cortex Analyst SQL runs as the caller.
4. **The row access policy** filters rows inside the sales domain by the caller's role.

In [ ]:
%%sql -r agents
SHOW AGENTS IN SCHEMA MULTI_AGENT_MCP_DEMO.APP

In [ ]:
names = set(col(agents, "name"))
expected = {"SALES_AGENT", "SUPPORT_AGENT", "ORCHESTRATOR_AGENT"}
check("1a agents", expected <= names, f"agents in {FQ}: {sorted(names)}")

In [ ]:
def usage_on_agents(role):
    g = q(f"SHOW GRANTS TO ROLE {role}")
    # SHOW GRANTS reports agents as granted_on = CORTEX_AGENT
    on_agent = g[col(g, "granted_on").isin(["CORTEX_AGENT", "AGENT"]) & (col(g, "privilege") == "USAGE")]
    return sorted(n.split(".")[-1].strip('"') for n in col(on_agent, "name"))


grants = {role: usage_on_agents(role) for role in PERSONA_ROLES}
for role, agents_ in grants.items():
    print(f"{role:40s} USAGE on agents: {agents_}")
sales_ok = all("SALES_AGENT" in grants[r] and "SUPPORT_AGENT" not in grants[r] for r in PERSONA_ROLES[:2])
support_ok = "SUPPORT_AGENT" in grants[PERSONA_ROLES[2]] and "SALES_AGENT" not in grants[PERSONA_ROLES[2]]
front_door = all("ORCHESTRATOR_AGENT" in grants[r] for r in PERSONA_ROLES)
check("1b grants", sales_ok and support_ok and front_door,
      "every persona has the orchestrator; sales personas reach SALES_AGENT only, support reaches SUPPORT_AGENT only")

In [ ]:
%%sql -r rap_admin
SELECT COUNT(*) AS ROWS_VISIBLE FROM SALES_BOOKINGS

In [ ]:
%%sql -r rap_emea
EXECUTE USING POLICY_CONTEXT(CURRENT_ROLE => 'MULTI_AGENT_MCP_DEMO_SALES_EMEA')
  AS SELECT COUNT(*) AS ROWS_VISIBLE, COUNT(DISTINCT THEATER) AS THEATERS FROM SALES_BOOKINGS

In [ ]:
%%sql -r rap_support
EXECUTE USING POLICY_CONTEXT(CURRENT_ROLE => 'MULTI_AGENT_MCP_DEMO_SUPPORT')
  AS SELECT COUNT(*) AS ROWS_VISIBLE FROM SALES_BOOKINGS

In [ ]:
n_admin = int(col(rap_admin, "rows_visible")[0])
n_emea = int(col(rap_emea, "rows_visible")[0])
n_support = int(col(rap_support, "rows_visible")[0])
check("1c row_access_policy", (n_admin, n_emea, n_support) == (44, 16, 0),
      f"rows visible: admin {n_admin} (all 11 territories x 4 quarters), EMEA seller {n_emea}, support {n_support} (fail closed)")

---
## Section 2 - One front door: `ORCHESTRATOR_MCP`

A Snowflake-managed MCP server with exactly one tool, `northwind_assistant`, of type
`CORTEX_AGENT_RUN` on `ORCHESTRATOR_AGENT`. No `SYSTEM_EXECUTE_SQL`: every MCP client gets the same
single front door. An MCP session runs as the caller's `DEFAULT_ROLE`, so each persona user reaches
exactly what Section 1 showed.

There is no ALTER for an MCP server specification, and `CREATE OR REPLACE` drops every grant on the
server, so the grants are reapplied right after.

In [ ]:
%%sql -r create_orchestrator_mcp
CREATE OR REPLACE MCP SERVER MULTI_AGENT_MCP_DEMO.APP.ORCHESTRATOR_MCP
  FROM SPECIFICATION $$
tools:
  - title: "Northwind Assistant"
    name: "northwind_assistant"
    type: "CORTEX_AGENT_RUN"
    identifier: "MULTI_AGENT_MCP_DEMO.APP.ORCHESTRATOR_AGENT"
    description: >-
      Governed answers about Northwind: sales new bookings, targets, attainment and
      pipeline by fiscal quarter, theater and territory; and product support how-to
      and troubleshooting (SSO, passwords, API tokens, 401/403 errors, billing and
      seats, exports, dashboard performance, audit logs). Pass the user's question
      as a self-contained message. Answers cover only data the caller is entitled to.
    annotations:
      readOnlyHint: true
      destructiveHint: false
      idempotentHint: false
      openWorldHint: false
$$

In [ ]:
# USAGE on the server allows tools/list. tools/call also needs USAGE on the agent
# and everything behind it, which setup.sql Part 7 already decides.
for role in PERSONA_ROLES:
    session.sql(f"GRANT USAGE ON MCP SERVER {FQ}.ORCHESTRATOR_MCP TO ROLE {role}").collect()


def mcp_usage_roles(server):
    g = q(f"SHOW GRANTS ON MCP SERVER {FQ}.{server}")
    return sorted(col(g, "grantee_name")[col(g, "privilege") == "USAGE"])


roles = mcp_usage_roles("ORCHESTRATOR_MCP")
desc = q(f"DESCRIBE MCP SERVER {FQ}.ORCHESTRATOR_MCP")
spec = " ".join(str(v) for v in desc.astype(str).values.ravel())
check("2 orchestrator_mcp",
      set(PERSONA_ROLES) <= set(roles) and "CORTEX_AGENT_RUN" in spec and "northwind_assistant" in spec,
      f"ORCHESTRATOR_MCP: one CORTEX_AGENT_RUN tool northwind_assistant; USAGE: {roles}")

---
## Section 3 - The alternative: one agent that inherits the specialists' tools

[Agent toolsets](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-toolsets)
let one agent reference others by name. At run time Snowflake checks the caller's USAGE on each
referenced agent and unions their tools (`sales_analyst`, `support_kb`) into the caller. One
orchestrator, one context and one budget answer the question; the specialists' own instructions,
models and budgets never run. A reference the caller has no USAGE on is **skipped silently**, so the
refusal wording lives in `TOOLSET_AGENT`'s instructions.

`TOOLSET_MCP` exposes it with the same tool name as the hub, so a client only changes the server URL.

In [ ]:
%%sql -r create_toolset_agent
CREATE OR REPLACE AGENT MULTI_AGENT_MCP_DEMO.APP.TOOLSET_AGENT
  COMMENT = 'Single agent that inherits the sales and support specialists'' tools through agent_toolset.'
  PROFILE = '{"display_name": "Northwind Assistant (toolset)"}'
  COPY GRANTS
  FROM SPECIFICATION
$$
models:
  orchestration: claude-sonnet-4-6
orchestration:
  budget:
    seconds: 300
    tokens: 100000
instructions:
  orchestration: |
    Answer only from your tools. Never answer company questions from your own knowledge.
    - sales_analyst: new bookings, targets, attainment, pipeline by fiscal quarter
      (FY25-Q1 to FY25-Q4), theater (Americas, EMEA, APAC) and territory.
    - support_kb: product how-to and troubleshooting: SSO, passwords, API tokens,
      401/403 errors, billing and seats, exports, dashboard performance, audit logs.
    If a question spans both domains, use both and answer each part under its own heading.
    If a question, or part of one, needs a tool you do not have, do not guess and do not
    use another tool instead. For that part reply exactly: "This type of question is not
    available at your access level. Contact your administrator for access."
  response: |
    Lead with the number. Values are in USD millions; format as $X.XM. Always state the
    fiscal quarter and theater you answered for. Use a chart for trends and comparisons
    across quarters or theaters. For support answers, give steps and cite the article ID.
  sample_questions:
    - question: "Show new bookings by fiscal quarter and theater as a chart"
    - question: "How do I set up single sign-on?"
tools:
  - tool_spec:
      type: agent_toolset
      name: sales_tools
  - tool_spec:
      type: agent_toolset
      name: support_tools
tool_resources:
  sales_tools:
    agent_name: MULTI_AGENT_MCP_DEMO.APP.SALES_AGENT
  support_tools:
    agent_name: MULTI_AGENT_MCP_DEMO.APP.SUPPORT_AGENT
$$

In [ ]:
%%sql -r create_toolset_mcp
CREATE OR REPLACE MCP SERVER MULTI_AGENT_MCP_DEMO.APP.TOOLSET_MCP
  FROM SPECIFICATION $$
tools:
  - title: "Northwind Assistant"
    name: "northwind_assistant"
    type: "CORTEX_AGENT_RUN"
    identifier: "MULTI_AGENT_MCP_DEMO.APP.TOOLSET_AGENT"
    description: >-
      Governed answers about Northwind: sales new bookings, targets, attainment and
      pipeline by fiscal quarter, theater and territory; and product support how-to
      and troubleshooting (SSO, passwords, API tokens, 401/403 errors, billing and
      seats, exports, dashboard performance, audit logs). Pass the user's question
      as a self-contained message. Answers cover only data the caller is entitled to.
    annotations:
      readOnlyHint: true
      destructiveHint: false
      idempotentHint: false
      openWorldHint: false
$$

In [ ]:
# Everyone may ask. What each persona's run inherits is decided by the existing
# USAGE grants on SALES_AGENT / SUPPORT_AGENT (setup.sql Part 7).
for role in PERSONA_ROLES:
    session.sql(f"GRANT USAGE ON AGENT {FQ}.TOOLSET_AGENT TO ROLE {role}").collect()
    session.sql(f"GRANT USAGE ON MCP SERVER {FQ}.TOOLSET_MCP TO ROLE {role}").collect()

names = set(col(q(f"SHOW AGENTS IN SCHEMA {FQ}"), "name"))
roles = mcp_usage_roles("TOOLSET_MCP")
check("3 toolset", "TOOLSET_AGENT" in names and set(PERSONA_ROLES) <= set(roles),
      f"TOOLSET_AGENT inherits sales_tools + support_tools; TOOLSET_MCP USAGE: {roles}")

---
## Section 4 - The persona boundary, seen from an MCP client

This is the traffic Claude, ChatGPT or Cursor would send: JSON-RPC `tools/list` and `tools/call`
to `https://<account>/api/v2/databases/MULTI_AGENT_MCP_DEMO/schemas/APP/mcp-servers/<server>`,
signed in as each persona user with a key-pair JWT (`../app/snowflake_mcp.py`). The server cannot
tell this client from any other. Same server, same tool, same question; only the signed-in user changes.

Expected FY25-Q4 bookings: Americas ~$59.6M, EMEA ~$38.4M, APAC ~$22.1M (total $120.0M).
The six calls run in parallel and take a minute or two. Needs the persona keys `./demo.sh setup`
wrote, so in Workspaces this section reports SKIP.

In [ ]:
sys.path.insert(0, str(Path.cwd().parent / "app"))
org_account = session.sql("SELECT CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME()").collect()[0][0]
os.environ.setdefault("SNOWFLAKE_ACCOUNT", org_account)
from personas import PERSONAS, SERVERS  # noqa: E402
from snowflake_mcp import KeyPairAuth, SnowflakeMcpClient, mcp_server_url, parse_agent_result  # noqa: E402

KEYS = {k: os.path.expanduser(os.environ.get(p["key_env"], "")) for k, p in PERSONAS.items()}
HAVE_KEYS = all(path and os.path.exists(path) for path in KEYS.values())
print(f"account {org_account}; persona keys {'found' if HAVE_KEYS else 'missing (run locally after ./demo.sh setup)'}")


def ask(persona, arch, question=QUESTION):
    p = PERSONAS[persona]
    url = mcp_server_url(org_account, DB, SCHEMA, SERVERS[arch]["server"])
    client = SnowflakeMcpClient(url, KeyPairAuth(org_account, p["user"], KEYS[persona]))
    tools, _ = client.list_tools()
    started = time.time()
    result, _ = client.call_tool(tools[0]["name"], {"text": question})
    parsed = parse_agent_result(result)
    return {"persona": persona, "arch": arch, "tools": [t["name"] for t in tools],
            "answer": parsed["answer"], "denied": any(d["denied"] for d in parsed["delegations"]),
            "delegations": [d["tool"] for d in parsed["delegations"]],
            "seconds": round(time.time() - started, 1)}

In [ ]:
THEATERS = {"Americas": 59.56, "EMEA": 38.35, "APAC": 22.09}


def mentions(text, value, tol=0.06):
    return any(abs(float(n) - value) <= tol for n in re.findall(r"\d+\.\d+", text))


if not HAVE_KEYS:
    print("CHECK 4 persona_boundary: SKIP | persona keys not on this machine; run ./demo.sh build locally")
else:
    jobs = [(p, a) for a in SERVERS for p in PERSONAS]
    with ThreadPoolExecutor(max_workers=len(jobs)) as pool:
        runs = list(pool.map(lambda job: ask(*job), jobs))
    results = pd.DataFrame(runs)
    display(results[["arch", "persona", "tools", "delegations", "denied", "seconds"]])
    for r in runs:
        print(f"\n--- {r['arch']} / {r['persona']} ({r['seconds']}s)\n{r['answer'][:600]}")

    by = {(r["persona"], r["arch"]): r for r in runs}
    one_tool = all(r["tools"] == ["northwind_assistant"] for r in runs)
    failures = []
    for arch in SERVERS:
        g, e, s = by[("sales_global", arch)], by[("sales_emea", arch)], by[("support", arch)]
        if not all(mentions(g["answer"], v) for v in THEATERS.values()):
            failures.append(f"{arch}: global exec missing a theater total")
        if not mentions(e["answer"], THEATERS["EMEA"]) or any(
                mentions(e["answer"], THEATERS[t]) for t in ("Americas", "APAC")):
            failures.append(f"{arch}: EMEA seller not limited to EMEA")
        if any(mentions(s["answer"], v) for v in THEATERS.values()):
            failures.append(f"{arch}: support rep received sales numbers")
        if arch == "hub" and not s["denied"]:
            failures.append("hub: no NOT_ENTITLED from ask_sales for the support rep")
    check("4 persona_boundary", one_tool and not failures,
          "one tool for everyone on both servers; global sees all theaters, EMEA sees EMEA only, "
          "support gets no sales numbers (hub: NOT_ENTITLED)" if not failures else "; ".join(failures))

The same boundary without MCP: call `ORCHESTRATOR_AGENT` with SQL `DATA_AGENT_RUN`, signed in as
each persona user (key-pair JWT). This is the check to run when you suspect the client rather than
the grants. Running it as yourself would prove nothing: the run authorizes against *your* default role.

In [ ]:
from snowflake.snowpark import Session as _Session


def ask_sql(persona, question=QUESTION):
    p = PERSONAS[persona]
    s = _Session.builder.configs({"account": org_account, "user": p["user"],
                                  "authenticator": "SNOWFLAKE_JWT", "private_key_file": KEYS[persona]}).create()
    try:
        role = s.sql("SELECT CURRENT_ROLE()").collect()[0][0]
        body = json.dumps({"messages": [{"role": "user", "content": [{"type": "text", "text": question}]}]})
        raw = s.sql("SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(?, ?) AS RESP",
                    params=[f"{FQ}.ORCHESTRATOR_AGENT", body]).collect()[0]["RESP"]
    finally:
        s.close()
    resp = json.loads(raw)
    text = "\n".join(b.get("text", "") for b in resp.get("content", []) if b.get("type") == "text")
    return {"persona": persona, "role": role, "answer": text}


if not HAVE_KEYS:
    print("CHECK 4b persona_boundary_sql: SKIP | persona keys not on this machine; run ./demo.sh build locally")
else:
    with ThreadPoolExecutor(max_workers=len(PERSONAS)) as pool:
        sql_runs = {r["persona"]: r for r in pool.map(ask_sql, PERSONAS)}
    for r in sql_runs.values():
        print(f"\n--- SQL / {r['persona']} (role {r['role']})\n{r['answer'][:500]}")
    g, e, s = (sql_runs[k]["answer"] for k in ("sales_global", "sales_emea", "support"))
    roles_ok = all(sql_runs[k]["role"] == PERSONAS[k]["role"] for k in PERSONAS)
    ok = (roles_ok and all(mentions(g, v) for v in THEATERS.values())
          and mentions(e, THEATERS["EMEA"]) and not any(mentions(e, THEATERS[t]) for t in ("Americas", "APAC"))
          and not any(mentions(s, v) for v in THEATERS.values()))
    check("4b persona_boundary_sql", ok,
          "DATA_AGENT_RUN as each persona (session role = persona DEFAULT_ROLE): "
          "same answers as over MCP")

---
## Section 5 - Score both architectures on the same five questions

[Cortex Agent evaluations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-evaluations)
run each question through the agent and score the trace: `answer_correctness` and
`logical_consistency` (GA), plus `tool_selection_accuracy` (Preview) for the toolset agent, all
pinned to v3. The expected tools differ: the hub calls the `ask_sales` / `ask_support` bridges, the
toolset agent calls the inherited `sales_analyst` / `support_kb` directly. Evaluations run as the
admin role (they do not impersonate personas), so these score routing and answers, not the boundary.

Both runs take several minutes on serverless tasks. Set `DEMO_SKIP_EVALS=1`
(`./demo.sh build --yes --skip-evals`) to skip this section.

In [ ]:
%%sql -r eval_format
CREATE FILE FORMAT IF NOT EXISTS EVAL_YAML_FORMAT
  TYPE = CSV FIELD_DELIMITER = NONE RECORD_DELIMITER = '\n' SKIP_HEADER = 0
  FIELD_OPTIONALLY_ENCLOSED_BY = NONE ESCAPE_UNENCLOSED_FIELD = NONE

In [ ]:
%%sql -r eval_stage
CREATE STAGE IF NOT EXISTS EVAL_CONFIG FILE_FORMAT = EVAL_YAML_FORMAT

In [ ]:
%%sql -r routing_eval_table
CREATE TABLE IF NOT EXISTS ROUTING_EVAL_INPUT (INPUT_QUERY VARCHAR, GROUND_TRUTH VARIANT)

In [ ]:
%%sql -r routing_eval_rows
INSERT OVERWRITE INTO ROUTING_EVAL_INPUT
SELECT q, PARSE_JSON(gt) FROM VALUES
  ('What were total new bookings in FY25-Q4 by theater?',
   '{"ground_truth_output": "FY25-Q4 new bookings: Americas about $59.6M, EMEA about $38.4M, APAC about $22.1M, total $120.0M. Values within 0.1M are acceptable.", "ground_truth_invocations": [{"tool_name": "ask_sales"}]}'),
  ('What was EMEA attainment against target in FY25-Q3?',
   '{"ground_truth_output": "EMEA FY25-Q3 new bookings were about $35.2M against a $35.2M target, attainment about 99.9%.", "ground_truth_invocations": [{"tool_name": "ask_sales"}]}'),
  ('How do I set up single sign-on?',
   '{"ground_truth_output": "Go to Settings > Security > SSO, choose the identity provider, paste its metadata URL, map the SAML NameID to the email attribute, and test with one admin before requiring SSO for all members. Cites KB-001.", "ground_truth_invocations": [{"tool_name": "ask_support"}]}'),
  ('Why is the API returning 403 for a valid token?',
   '{"ground_truth_output": "A 403 means the token is valid but lacks the scope for the endpoint; create a token with the right scope. Cites KB-003.", "ground_truth_invocations": [{"tool_name": "ask_support"}]}'),
  ('What was total pipeline in FY25-Q4, and how do I schedule a recurring export?',
   '{"ground_truth_output": "Total FY25-Q4 pipeline was $360.0M. Recurring exports are set up under Settings > Data > Scheduled Exports, daily or weekly, as CSV or JSON. Both parts must be answered.", "ground_truth_invocations": [{"tool_name": "ask_sales"}, {"tool_name": "ask_support"}]}')
  AS t(q, gt)

In [ ]:
%%sql -r routing_eval_dataset
EXECUTE IMMEDIATE $$
BEGIN
  CALL SYSTEM$CREATE_EVALUATION_DATASET(
    'Cortex Agent',
    'MULTI_AGENT_MCP_DEMO.APP.ROUTING_EVAL_INPUT',
    'MULTI_AGENT_MCP_DEMO.APP.ROUTING_EVAL_DATASET',
    OBJECT_CONSTRUCT('query_text', 'INPUT_QUERY', 'expected_tools', 'GROUND_TRUTH'));
  RETURN 'dataset created';
EXCEPTION
  WHEN OTHER THEN RETURN 'dataset already exists: ' || SQLERRM;
END;
$$

In [ ]:
%%sql -r toolset_eval_table
CREATE TABLE IF NOT EXISTS TOOLSET_EVAL_INPUT (INPUT_QUERY VARCHAR, GROUND_TRUTH VARIANT)

In [ ]:
%%sql -r toolset_eval_rows
INSERT OVERWRITE INTO TOOLSET_EVAL_INPUT
SELECT q, PARSE_JSON(gt) FROM VALUES
  ('What were total new bookings in FY25-Q4 by theater?',
   '{"ground_truth_output": "FY25-Q4 new bookings: Americas about $59.6M, EMEA about $38.4M, APAC about $22.1M, total $120.0M. Values within 0.1M are acceptable.", "ground_truth_invocations": [{"tool_name": "sales_analyst"}]}'),
  ('What was EMEA attainment against target in FY25-Q3?',
   '{"ground_truth_output": "EMEA FY25-Q3 new bookings were about $35.2M against a $35.2M target, attainment about 99.9%.", "ground_truth_invocations": [{"tool_name": "sales_analyst"}]}'),
  ('How do I set up single sign-on?',
   '{"ground_truth_output": "Go to Settings > Security > SSO, choose the identity provider, paste its metadata URL, map the SAML NameID to the email attribute, and test with one admin before requiring SSO for all members. Cites KB-001.", "ground_truth_invocations": [{"tool_name": "support_kb"}]}'),
  ('Why is the API returning 403 for a valid token?',
   '{"ground_truth_output": "A 403 means the token is valid but lacks the scope for the endpoint; create a token with the right scope. Cites KB-003.", "ground_truth_invocations": [{"tool_name": "support_kb"}]}'),
  ('What was total pipeline in FY25-Q4, and how do I schedule a recurring export?',
   '{"ground_truth_output": "Total FY25-Q4 pipeline was $360.0M. Recurring exports are set up under Settings > Data > Scheduled Exports, daily or weekly, as CSV or JSON. Both parts must be answered.", "ground_truth_invocations": [{"tool_name": "sales_analyst"}, {"tool_name": "support_kb"}]}')
  AS t(q, gt)

In [ ]:
%%sql -r toolset_eval_dataset
EXECUTE IMMEDIATE $$
BEGIN
  CALL SYSTEM$CREATE_EVALUATION_DATASET(
    'Cortex Agent',
    'MULTI_AGENT_MCP_DEMO.APP.TOOLSET_EVAL_INPUT',
    'MULTI_AGENT_MCP_DEMO.APP.TOOLSET_EVAL_DATASET',
    OBJECT_CONSTRUCT('query_text', 'INPUT_QUERY', 'expected_tools', 'GROUND_TRUTH'));
  RETURN 'dataset created';
EXCEPTION
  WHEN OTHER THEN RETURN 'dataset already exists: ' || SQLERRM;
END;
$$

In [ ]:
%%sql -r routing_eval_config
COPY INTO @EVAL_CONFIG/routing_eval.yaml
FROM (SELECT $$evaluation:
  agent_params:
    agent_name: "MULTI_AGENT_MCP_DEMO.APP.ORCHESTRATOR_AGENT"
    agent_type: "CORTEX AGENT"
    agent_version: "LIVE"
  run_params:
    label: "routing"
    description: "Does the orchestrator reach the right specialist and answer correctly?"
  source_metadata:
    type: "dataset"
    dataset_name: "MULTI_AGENT_MCP_DEMO.APP.ROUTING_EVAL_DATASET"
metrics:
  - name: "answer_correctness"
    version: "v3"
  - name: "logical_consistency"
    version: "v3"
$$)
FILE_FORMAT = (TYPE = CSV COMPRESSION = NONE RECORD_DELIMITER = NONE FIELD_DELIMITER = NONE
               FIELD_OPTIONALLY_ENCLOSED_BY = NONE ESCAPE_UNENCLOSED_FIELD = NONE)
OVERWRITE = TRUE SINGLE = TRUE HEADER = FALSE

In [ ]:
%%sql -r toolset_eval_config
COPY INTO @EVAL_CONFIG/toolset_eval.yaml
FROM (SELECT $$evaluation:
  agent_params:
    agent_name: "MULTI_AGENT_MCP_DEMO.APP.TOOLSET_AGENT"
    agent_type: "CORTEX AGENT"
    agent_version: "LIVE"
  run_params:
    label: "toolset-routing"
    description: "Does the toolset agent pick the right inherited tool and answer correctly?"
  source_metadata:
    type: "dataset"
    dataset_name: "MULTI_AGENT_MCP_DEMO.APP.TOOLSET_EVAL_DATASET"
metrics:
  - name: "answer_correctness"
    version: "v3"
  - name: "logical_consistency"
    version: "v3"
  - name: "tool_selection_accuracy"
    version: "v3"
$$)
FILE_FORMAT = (TYPE = CSV COMPRESSION = NONE RECORD_DELIMITER = NONE FIELD_DELIMITER = NONE
               FIELD_OPTIONALLY_ENCLOSED_BY = NONE ESCAPE_UNENCLOSED_FIELD = NONE)
OVERWRITE = TRUE SINGLE = TRUE HEADER = FALSE

In [ ]:
EVALS = {"ORCHESTRATOR_AGENT": ("routing-baseline", "routing_eval.yaml"),
         "TOOLSET_AGENT": ("toolset-baseline", "toolset_eval.yaml")}


def evaluation(job, run_name, config):
    return session.sql(f"CALL EXECUTE_AI_EVALUATION('{job}', OBJECT_CONSTRUCT('run_name', '{run_name}'), "
                       f"'@{FQ}.EVAL_CONFIG/{config}')").collect()


if SKIP_EVALS:
    print("CHECK 5 evaluations: SKIP | DEMO_SKIP_EVALS=1")
else:
    for run_name, config in EVALS.values():
        try:
            evaluation("DELETE", run_name, config)      # replace a previous run of the same name
        except Exception:
            pass                                        # no previous run
        evaluation("START", run_name, config)
    # PARTIALLY_COMPLETED can be transient while the last metric is still computing,
    # so wait for COMPLETED (or CANCELLED) until the deadline.
    done = {"COMPLETED", "CANCELLED"}
    status, deadline = {}, time.time() + 25 * 60
    while time.time() < deadline:
        status = {agent: evaluation("STATUS", *cfg)[0]["STATUS"] for agent, cfg in EVALS.items()}
        print(time.strftime("%H:%M:%S"), status)
        if all(s in done for s in status.values()):
            break
        time.sleep(30)
    print(status)

In [ ]:
if not SKIP_EVALS:
    status = {agent: evaluation("STATUS", *cfg)[0]["STATUS"] for agent, cfg in EVALS.items()}
    frames = []
    for agent, (run_name, _) in EVALS.items():
        frames.append(q(f"""
            SELECT '{agent}' AS AGENT, METRIC_NAME, ROUND(AVG(EVAL_AGG_SCORE), 2) AS AVG_SCORE, COUNT(*) AS N
            FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA('{DB}', '{SCHEMA}', '{agent}', 'CORTEX AGENT', '{run_name}'))
            GROUP BY METRIC_NAME"""))
    scores = pd.concat(frames, ignore_index=True).sort_values(["METRIC_NAME", "AGENT"])
    display(scores)
    complete = all(s == "COMPLETED" for s in status.values())
    have_scores = all((scores["AGENT"] == agent).any() for agent in EVALS)
    summary = ", ".join(f"{r.AGENT} {r.METRIC_NAME}={r.AVG_SCORE} (n={r.N})" for r in scores.itertuples())
    check("5 evaluations", complete and have_scores, f"status {status}; {summary}")

---
## Section 6 - Demo click-path (15 minutes)

Talk-over: "One connector, one tool, for every AI client. Snowflake decides the answer for each person."

1. **App, 0-3 min** (`./demo.sh app`): switch personas in the sidebar. `tools/list` returns the same
   single tool each time; the client has no idea there are two specialists behind it.
2. **App, 3-6 min**: as the EMEA seller ask "What were total new bookings in FY25-Q4 by theater?". Open
   the tool call: the arguments Claude wrote, the JSON-RPC request, and under **Inside Snowflake** the
   orchestrator delegating to `ask_sales`. The answer is EMEA only.
3. **App, 6-9 min**: **Compare** personas on the same question (Section 4's results). For the support
   rep, `ask_sales` returns `NOT_ENTITLED`: grants refused it, no model was asked to.
4. **App, 9-12 min**: switch **Compare** to both architectures as the global exec and ask for a chart:
   native chart from the toolset, a re-drawn chart about three times slower from the hub.
5. **Notebook, 12-14 min**: Section 5's scores side by side, and the silent-skip trade-off.
6. **Snowsight, 14-15 min**: **AI & ML > Agents > ORCHESTRATOR_AGENT > Monitoring** shows each run
   attributed to the persona user. Then: point Claude Desktop or Cursor at the same URL with OAuth.

In [ ]:
for section, (status_, evidence) in CHECKS.items():
    print(f"{status_:4s}  {section:22s} {evidence[:150]}")